# Session 2 — Load validated DGAT predictions for the same lymph node

## Goal

Use the organizer-generated prediction matrix as the fast, reproducible participant workflow. We validate its provenance, checksum, spot barcodes, and 31-protein identifiers before creating the Session 3 handoff.

**These values are model predictions, not measured proteins.** Complete DGAT inference is retained at the end as an optional GPU reproducibility section.

## 1. Mount Drive(estimated_runtime 1min)

In [ ]:
REQUIREMENTS = []

from pathlib import Path
import importlib, importlib.util, json, os, shutil, subprocess, sys

SESSION_REQUIREMENTS = REQUIREMENTS
in_colab = importlib.util.find_spec("google.colab") is not None and Path("/content").is_dir()
if in_colab:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    repo_dir = Path("/content/ECCB-2026-Tutorial")
    tutorial_root = repo_dir / "hands-on_tutorial"
    if not (tutorial_root / "src" / "dgat_tutorial").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/osmanbeyoglulab/ECCB-2026-Tutorial.git", str(repo_dir)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo_dir), "fetch", "--depth", "1", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(repo_dir), "reset", "--hard", "origin/main"], check=True)
    drive_root = Path("/content/drive/MyDrive/ECCB2026")
    manifest_path = drive_root / "asset_manifest.json"
    if not manifest_path.is_file(): raise FileNotFoundError(f"Missing {manifest_path}; run Session 0.")
    manifest = json.loads(manifest_path.read_text())
    drive_assets = drive_root / "assets" / "DGAT_assets"
    local_assets = tutorial_root / "external" / "DGAT_assets"
    local_data = local_assets / "data"
    local_data.mkdir(parents=True, exist_ok=True)
    for filename in ("V1_Human_Lymph_Node_filtered_feature_bc_matrix.h5", "V1_Human_Lymph_Node_manual_GC_annot.csv"):
        source, destination = drive_assets / "data" / filename, local_data / filename
        expected = manifest["files"][filename]["bytes"]
        if not source.is_file() or source.stat().st_size != expected: raise IOError(f"Invalid Drive asset: {source}")
        if not destination.is_file() or destination.stat().st_size != expected: shutil.copy2(source, destination)
    source_spatial, destination_spatial = drive_assets / "data" / "spatial", local_data / "spatial"
    if not source_spatial.is_dir(): raise FileNotFoundError(f"Missing {source_spatial}")
    if destination_spatial.exists(): shutil.rmtree(destination_spatial)
    shutil.copytree(source_spatial, destination_spatial)
    os.environ["DGAT_TUTORIAL_STATE_DIR"] = str(drive_root / "state")
    dgat_dir = tutorial_root / "external" / "DGAT"
    if not (dgat_dir / "utils" / "Preprocessing.py").is_file():
        dgat_dir.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/osmanbeyoglulab/DGAT.git", str(dgat_dir)], check=True)
    missing = [spec for module, spec in SESSION_REQUIREMENTS if importlib.util.find_spec(module) is None]
    if missing:
        wheelhouse = drive_root / "wheelhouse" / f"py{sys.version_info.major}{sys.version_info.minor}"
        cmd = [sys.executable, "-m", "pip", "install", "-q", *missing]
        if wheelhouse.is_dir(): cmd[4:4] = ["--find-links", str(wheelhouse)]
        subprocess.run(cmd, check=True); importlib.invalidate_caches()
else:
    tutorial_root = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "src" / "dgat_tutorial").is_dir()), None)
    if tutorial_root is None: raise FileNotFoundError("Could not locate hands-on_tutorial.")
    drive_root = None

os.chdir(tutorial_root)
sys.path.insert(0, str(tutorial_root / "src"))
from dgat_tutorial.checkpoints import tutorial_paths, write_checkpoint
paths = tutorial_paths(tutorial_root)
print("Tutorial root:", paths.root)
print("Completed checkpoints:", sorted(p.name for p in paths.checkpoints.glob("session_*/part_*.json")) or "none")


## Step 1 — Reload the exact Session 1 spot contract(estimated_runtime 7sec)

The precomputed rows are accepted only if they contain the same unique barcodes in the same order as the model-compatible Session 1 checkpoint.

In [ ]:
import hashlib, json, numpy as np, pandas as pd, shutil, subprocess, sys
from scipy import sparse
spots = pd.read_csv(paths.processed_data / "filtered_spots.csv", index_col=0)
rna_matrix = sparse.load_npz(paths.processed_data / "filtered_rna_counts.npz")
rna_genes = pd.read_csv(paths.processed_data / "filtered_rna_genes.csv")["gene"].astype(str).tolist()
gc = pd.read_csv(paths.processed_data / "germinal_center_labels.csv", index_col=0).iloc[:,0].astype(str).map({"True":True,"False":False,True:True,False:False})
if not spots.index.is_unique: raise ValueError("Session 1 spot barcodes are not unique")
if not spots.index.equals(gc.index) or rna_matrix.shape != (len(spots), len(rna_genes)): raise ValueError("Session 1 checkpoint dimensions/IDs differ")
print(f"Session 1 contract: {len(spots)} spots × {len(rna_genes)} observed genes")

## Step 2 — Load and validate the precomputed prediction matrix(estimated_runtime 2sec)

This is the default hands-on path. It performs no neural-network inference. The matrix must name this lymph-node sample, match its own recorded checksum, contain finite values, and exactly match both the Session 1 barcodes and DGAT's ordered 31-protein output contract.

In [ ]:
from dgat_tutorial.dgat import load_prediction_metadata, load_prediction_table
from dgat_tutorial.alignment import require_exact_identifiers

precomputed_dir = (drive_root / "assets/DGAT_assets/data") if in_colab else (paths.root / "external/DGAT_assets/data")
precomputed_path = precomputed_dir / "V1_Human_Lymph_Node_DGAT_predicted_proteins.csv"
precomputed_metadata_path = precomputed_path.with_suffix(".metadata.json")
if not precomputed_path.is_file() or not precomputed_metadata_path.is_file():
    raise FileNotFoundError("Missing validated lymph-node predictions; rerun Session 0 after the organizer build is complete.")

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b""): h.update(chunk)
    return h.hexdigest()

predicted = load_prediction_table(precomputed_path)
metadata = load_prediction_metadata(precomputed_path)
if metadata is None or metadata.get("dataset") != "V1_Human_Lymph_Node": raise ValueError("Prediction sidecar names a different or missing sample")
recorded_sha = metadata.get("artifact_sha256") or metadata.get("prediction_sha256")
if recorded_sha and recorded_sha != sha256(precomputed_path): raise ValueError("Prediction checksum differs from its metadata sidecar")
require_exact_identifiers(spots.index, predicted.index, left_name="Session 1 spots", right_name="precomputed prediction spots")
dgat_dir = paths.root / "external/DGAT"
common_proteins = [x.strip() for x in (dgat_dir/"resources/common_protein_31.txt").read_text().splitlines() if x.strip()]
require_exact_identifiers(pd.Index(common_proteins), predicted.columns, left_name="DGAT protein contract", right_name="precomputed prediction proteins")
if not np.isfinite(predicted.to_numpy(dtype=float)).all(): raise ValueError("Prediction matrix contains NaN or infinite values")
for marker in ("CD19","CXCR5","PDCD1"):
    if marker not in predicted.columns: raise KeyError(f"Missing required marker: {marker}")
print(metadata.get("evaluation_scope", metadata.get("evaluation_note", "Inferred proteins are predictions, not measurements.")))
print(f"Validated precomputed matrix: {predicted.shape[0]} spots × {predicted.shape[1]} proteins")
display(predicted[["CD19","CXCR5","PDCD1"]].head())

## Step 3 — Create the restart-safe Session 3 handoff(estimated_runtime 1sec)

The validated CSV and its original organizer metadata are copied without changing barcode or protein order. The checkpoint records the distributed artifact checksum.

In [ ]:
prediction_path = paths.processed_data / "predicted_proteins.csv"
metadata_path = prediction_path.with_suffix(".metadata.json")
shutil.copy2(precomputed_path, prediction_path)
shutil.copy2(precomputed_metadata_path, metadata_path)
reloaded = load_prediction_table(prediction_path)
require_exact_identifiers(spots.index, reloaded.index, left_name="Session 1 spots", right_name="Session 3 handoff spots")
require_exact_identifiers(pd.Index(common_proteins), reloaded.columns, left_name="DGAT protein contract", right_name="Session 3 handoff proteins")
manifest = write_checkpoint("2.1",[prediction_path,metadata_path],summary={"sample":"V1_Human_Lymph_Node","spots":len(reloaded),"proteins":reloaded.shape[1],"prediction_sha256":sha256(prediction_path),"workflow":"precomputed_default"},start=paths.root)
print("Checkpoint:",manifest)

## Check

Confirm that the printed workflow contains the expected lymph-node spot count and exactly 31 proteins. CD19, CXCR5, and PDCD1 must be present. Passing these checks establishes artifact integrity and alignment—not protein-level accuracy.

## Next steps

Session 3 treats this table as an inferred molecular view and evaluates spatial and anatomical coherence—not pointwise protein accuracy.

## Optional reproducibility — rerun complete DGAT inference

This section is **not part of the default hands-on workflow**. Set `RUN_OPTIONAL_FULL_INFERENCE=True` only in a GPU runtime with sufficient time and memory. It downloads the released weights, installs the graph dependencies, reruns the 11,535-gene encoder and 31-protein decoder, and compares the rerun numerically with the distributed artifact.

In [ ]:
RUN_OPTIONAL_FULL_INFERENCE = False
if not RUN_OPTIONAL_FULL_INFERENCE:
    print("Optional inference skipped. The validated precomputed matrix remains the participant handoff.")

In [ ]:
if RUN_OPTIONAL_FULL_INFERENCE:
    optional_specs=["muon==0.1.7","mudata==0.3.1","torch-geometric==2.6.1","gdown==6.1.0"]
    subprocess.run([sys.executable,"-m","pip","install","-q",*optional_specs],check=True); importlib.invalidate_caches()
    from importlib.metadata import version as package_version
    if package_version("mudata") != "0.3.1":
        raise RuntimeError("Expected mudata==0.3.1. Restart the runtime before optional inference.")
    if in_colab:
        env=os.environ.copy(); env["DGAT_ASSET_DIR"]=str(drive_root/"assets/DGAT_assets"); env["DGAT_PRECOMPUTED_DIR"]=str(drive_root/"organizer/lymph_node_prediction_build")
        subprocess.run(["bash","scripts/download_dgat_assets.sh","--dataset","V1_Human_Lymph_Node","--include-model"],cwd=tutorial_root,env=env,check=True)

    from anndata import AnnData
    sys.path.insert(0,str(dgat_dir))
    from utils.Preprocessing import fill_genes, preprocess_ST
    from Model.Train_and_Predict import protein_predict
    common_genes=[x.strip() for x in (dgat_dir/"resources/common_gene_11535.txt").read_text().splitlines() if x.strip()]
    adata=AnnData(X=rna_matrix.astype(np.float32),obs=spots.copy(),var=pd.DataFrame(index=rna_genes))
    adata.obsm["spatial"]=spots[["x","y"]].to_numpy(float); adata.uns["name"]="V1_Human_Lymph_Node"
    adata=fill_genes(adata,common_genes); preprocess_ST(adata)
    model_root=(drive_root/"assets/DGAT_assets/DGAT_pretrained_models" if in_colab else paths.root/"external/DGAT_assets/DGAT_pretrained_models")
    rerun_adata=protein_predict(adata,common_genes,common_proteins,str(model_root),str(paths.processed_data/"pyg_lymph_node_optional"))
    rerun=pd.DataFrame(np.asarray(rerun_adata.X),index=rerun_adata.obs_names,columns=common_proteins)
    require_exact_identifiers(predicted.index,rerun.index,left_name="precomputed spots",right_name="optional rerun spots")
    comparison=pd.DataFrame({"pearson_r":[predicted[p].corr(rerun[p]) for p in common_proteins],"mean_absolute_difference":[float((predicted[p]-rerun[p]).abs().mean()) for p in common_proteins]},index=common_proteins)
    optional_path=paths.results/"session02_optional_inference_comparison.csv"; comparison.to_csv(optional_path)
    display(comparison); print("Optional comparison:",optional_path)